# Validation technique du déploiement d'AutoGluon Tabular

## Projet VolatiChainXplorerAI - Sprint 2

### 1. Contexte

À l'issue du benchmark comparatif des solutions AutoML, **AutoGluon Tabular** a été retenu comme solution technologique privilégiée pour la prédiction de la volatilité du Bitcoin à 7 et 30 jours.

Cette sélection repose sur les performances prédictives obtenues face à PyCaret et à la baseline de persistance de la volatilité historique.

Toutefois, le choix d'une solution AutoML ne garantit pas, à lui seul, sa capacité à être intégrée dans une application.

Une validation technique complémentaire est donc nécessaire pour examiner les conditions d'exploitation des modèles précédemment entraînés.

### 2. Objectif

Ce notebook vise à **vérifier la faisabilité technique de l'intégration des modèles AutoGluon Tabular dans l'application VolatiChainXplorerAI**, sans procéder à un nouvel entraînement.

Il s'agit de confirmer que les modèles sauvegardés peuvent être chargés et utilisés indépendamment du notebook expérimental, tout en identifiant les contraintes techniques susceptibles d'influencer leur déploiement.

### 3. Périmètre des vérifications

Les vérifications portent sur cinq aspects :

1. **Compatibilité de l'environnement d'exécution :** identifier les versions de Python, d'AutoGluon et des principales dépendances nécessaires au fonctionnement des modèles.

2. **Disponibilité des artefacts :** vérifier la présence et l'accessibilité des modèles sauvegardés pour les horizons de 7 et 30 jours.

3. **Chargement des modèles :** confirmer la possibilité de restaurer les modèles entraînés à l'aide de `TabularPredictor.load()`, sans relancer leur apprentissage.

4. **Exécution des prédictions :** vérifier que les modèles restaurés peuvent produire des prédictions à partir de données conformes aux variables explicatives attendues.

5. **Analyse des contraintes de déploiement :** identifier les conditions techniques nécessaires à leur future intégration dans une application Streamlit, notamment les dépendances, les chemins d'accès, les ressources et la reproductibilité de l'environnement.

### 4. Résultats attendus

À l'issue de cette validation, les éléments suivants devront être documentés :

- La configuration de l'environnement d'exécution.
- La présence et le chargement effectif des modèles à 7 et 30 jours.
- La capacité à produire des prédictions sans nouvel entraînement.
- Les éventuelles difficultés techniques rencontrées et les solutions apportées.
- Les prérequis et recommandations pour l'intégration applicative.

### 5. Limites du périmètre

Ce notebook constitue une **preuve de faisabilité technique**, et non une validation définitive de la qualité prédictive ou de l'aptitude des modèles à être exploités en production.

L'entraînement, le benchmark et l'évaluation prédictive ont déjà été réalisés dans les notebooks précédents.

Le développement de l'interface Streamlit, l'industrialisation du pipeline et les tests applicatifs complets relèveront des étapes ultérieures du projet.

## 1. Compatibilité de l'environnement d'exécution 



In [14]:
import sys
import pandas as pd

from importlib.metadata import version

print("Python environment: ", sys.executable)
print("python version: ", sys.version.split()[0])
print("AutoGluon version :", version("autogluon.tabular"))
print("pandas version: ", pd.__version__)



Python environment:  d:\venvs\autogluon\Scripts\python.exe
python version:  3.12.9
AutoGluon version : 1.6.3
pandas version:  2.3.3


## 2. Vérifier la présence des modèles sauvegardés

Nous allons maintenant vérifier que les artefacts des modèles à 7 et 30 jours sont disponibles.

In [15]:
from pathlib import Path

# Racine du projet
project_root = Path.cwd().parent

# Répertoire des modèles sauvegardés
models_dir = (
    project_root 
    / "data" 
    / "exports" 
    / "autogluon_models"
)

predictor_7d_path = models_dir / "predictor_7d"
predictor_30d_path = models_dir / "predictor_30d"

print("Modèle 7 jours :", predictor_7d_path.is_dir())
print("Modèle 30 jours :", predictor_30d_path.is_dir())

print("\nChemin 7 jours :", predictor_7d_path)
print("Chemin 30 jours :", predictor_30d_path)

Modèle 7 jours : True
Modèle 30 jours : True

Chemin 7 jours : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports\autogluon_models\predictor_7d
Chemin 30 jours : d:\ProjectFolderDevAI_2025-2026\Certification BLOCS 2&3\Projet-VolatiChainXplorerAI\data\exports\autogluon_models\predictor_30d


### Charger les modèles à 7 et 30 jours sauvegardés

In [16]:
from autogluon.tabular import TabularPredictor

# Chargement des modèles sans nouvel entraînement
predictor_7d = TabularPredictor.load(
    str(predictor_7d_path)
)

predictor_30d = TabularPredictor.load(
    str(predictor_30d_path)
)

# Vérification
print("Modèle 7 jours chargé :", predictor_7d is not None)
print("Modèle 30 jours chargé :", predictor_30d is not None)

print("\nCible 7 jours :", predictor_7d.label)
print("Cible 30 jours :", predictor_30d.label)

Modèle 7 jours chargé : True
Modèle 30 jours chargé : True

Cible 7 jours : target_volatility_7d
Cible 30 jours : target_volatility_30d


## 4. Vérifier les prédictions après chargement

**Objectif** : Vérifier que les modèles restaurés peuvent produire des prédictions à partir des variables explicatives attendues.

Ce que nous voulons confirmer

- Chaque modèle produit cinq prédictions numériques.
- Les prédictions sont obtenues sans entraînement.
- Le modèle WeightedEnsemble_L2, retenu lors du benchmark, est bien exploitable.

In [18]:
splits_dir = (
    project_root 
    / "data" 
    / "exports" 
    / "datasets"
    / "benchmark_splits"
)

# Charger les données de validation
validation_7d = pd.read_csv(splits_dir / "validation_7d.csv")
validation_30d = pd.read_csv(splits_dir / "validation_30d.csv")

# Préparer 5 observations sans les cibles
X_sample_7d = validation_7d.drop(
    columns=["target_volatility_7d", "date"],
    errors="ignore"
).head(5)

X_sample_30d = validation_30d.drop(
    columns=["target_volatility_30d", "date"],
    errors="ignore"
).head(5)

# Prédire avec les modèles sauvegardés
predictions_7d = predictor_7d.predict(
    X_sample_7d,
    model="WeightedEnsemble_L2"
)

predictions_30d = predictor_30d.predict(
    X_sample_30d,
    model="WeightedEnsemble_L2"
)

print("Prédictions à 7 jours :")
display(predictions_7d)

print("Prédictions à 30 jours :")
display(predictions_30d)

Prédictions à 7 jours :


0    0.016211
1    0.016105
2    0.015588
3    0.015564
4    0.015739
Name: target_volatility_7d, dtype: float32

Prédictions à 30 jours :


0    0.018943
1    0.020395
2    0.019591
3    0.019334
4    0.018548
Name: target_volatility_30d, dtype: float32

## 5. Analyse des contraintes techniques de déploiement

### 5.1. Environnement d'exécution

Les expérimentations et les tests de restauration ont été réalisés sous Windows, dans un environnement virtuel Python indépendant.

| Composant | Configuration validée |
|---|---|
| Système d'exploitation | Windows 11 |
| Python | 3.12.9 |
| AutoGluon Tabular | 1.6.3 |
| Pandas | 2.3.3 |
| Environnement virtuel | `D:\venvs\autogluon` |
| Mode d'exécution | CPU |

L'installation d'AutoGluon a nécessité une adaptation du chemin de l'environnement virtuel afin de résoudre une difficulté liée à la longueur des chemins Windows (`WinError 206`).

Cette difficulté a été résolue et n'a pas empêché la réalisation des expérimentations.

### 5.2. Sauvegarde et restauration des modèles

Les deux prédicteurs entraînés sont conservés dans les répertoires suivants :

- `data/exports/datasets/autogluon_models/predictor_7d`
- `data/exports/datasets/autogluon_models/predictor_30d`

Les tests réalisés avec `TabularPredictor.load()` ont confirmé leur chargement dans un nouveau notebook, sans réentraînement.

Les modèles `WeightedEnsemble_L2` ont ensuite produit cinq prédictions numériques pour chaque horizon à partir d'observations issues des partitions de validation.

**Résultat : la réutilisation des modèles sauvegardés est techniquement validée dans l'environnement expérimental.**

### 5.3. Dépendances et reproductibilité

Le fonctionnement des prédicteurs nécessite un environnement compatible avec les bibliothèques utilisées lors de leur entraînement.

Pour garantir la reproductibilité, il conviendra de :

- Conserver les versions des principales dépendances Python.
- Documenter la procédure d'installation de l'environnement.
- Préserver l'intégralité des répertoires des prédicteurs AutoGluon.
- Prévoir des chemins d'accès configurables pour l'application.
- Vérifier le chargement des modèles dans tout nouvel environnement cible.

La compatibilité avec un autre système d'exploitation ou une infrastructure de production n'a pas encore été vérifiée.

### 5.4. Ressources matérielles

Les expérimentations ont été réalisées sur CPU.

Les mesures effectuées lors du benchmark concernent l'utilisation globale du système et ne permettent pas de déterminer précisément la mémoire requise par les modèles lors de leur chargement ou de leur utilisation.

Une évaluation complémentaire des besoins en mémoire, du stockage des artefacts et de la latence applicative pourra être réalisée lors de l'intégration.

### 5.5. Conditions d'intégration dans Streamlit

La future application devra permettre de :

1. Charger les prédicteurs à 7 et 30 jours sans relancer leur entraînement.
2. Fournir les 15 variables explicatives attendues, dans un format compatible.
3. Sélectionner le modèle `WeightedEnsemble_L2` pour chaque horizon.
4. Produire et afficher les prédictions de volatilité.
5. Contrôler les données d'entrée et gérer les éventuelles erreurs d'exécution.

Ces opérations relèveront de la phase de développement applicatif du Bloc 3.

## 6. Conclusion de la validation technique

Les vérifications réalisées confirment que les prédicteurs AutoGluon Tabular sauvegardés peuvent être restaurés et réutilisés pour produire des prédictions à 7 et 30 jours, sans nouvel entraînement.

**La faisabilité technique de leur réutilisation dans l'environnement Windows expérimental est donc démontrée.**

Cette validation constitue un premier élément favorable à leur future intégration dans VolatiChainXplorerAI.

Elle ne constitue toutefois pas une validation complète du déploiement applicatif ni de la fiabilité prédictive en production.

Les étapes ultérieures devront confirmer la reproductibilité de l'environnement cible, la gestion des dépendances, les ressources nécessaires et le fonctionnement de la chaîne de prédiction au sein de Streamlit.
